Bibles

In [1]:
# Data manupalation
import pandas as pd

# Data aquisition
import os
import glob

# BACEN data
from bcb import sgs

#Yfinance Date
import yfinance as yf

Import data

In [2]:
# Define the folder path and the specific test CSV file
data_folder = '../data'
file_name = 'planilha_201204.csv'
file_path = os.path.join(data_folder, file_name)

# Check if the file exists before attempting to read
if os.path.exists(file_path):
    df_test = pd.read_csv(file_path, sep=';', encoding='utf-8-sig')
    
    # Display basic metadata to verify the structure
    print(f"Successfully loaded file: {file_name}")
    print("-" * 30)
    
    # Check data types (crucial for checking if dates are strings or objects)
    print("Data Types and Non-Null Counts:")
    print(df_test.info())
    
    # Preview the first few rows
    print("\nFirst 5 rows of the dataset:")
    print(df_test.head())
    
    # Verify columns (check if your 8 income brackets are correctly loaded)
    print("\nColumn Names:")
    print(df_test.columns.tolist())
else:
    print(f"Error: The file '{file_name}' was not found in the '{data_folder}' directory.")

Successfully loaded file: planilha_201204.csv
------------------------------
Data Types and Non-Null Counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 497253 entries, 0 to 497252
Data columns (total 23 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   data_base                       497253 non-null  object 
 1   uf                              497253 non-null  object 
 2   tcb                             497253 non-null  object 
 3   sr                              0 non-null       float64
 4   cliente                         497253 non-null  object 
 5   ocupacao                        497253 non-null  object 
 6   cnae_secao                      497253 non-null  object 
 7   cnae_subclasse                  497253 non-null  object 
 8   porte                           497253 non-null  object 
 9   modalidade                      497253 non-null  object 
 10  origem                         

In [3]:
# 1. Define the columns that must be numeric
cols_to_process = ['vencido_acima_de_15_dias', 'carteira_inadimplida_arrastada' , 'ativo_problematico', 'carteira_ativa']

# 2. CLEANING PHASE: Fix commas and convert to float
for col in cols_to_process:
    # Convert to string, replace comma with dot, then convert to numeric
    # errors='coerce' will turn any garbage text into NaN (null)
    df_test[col] = pd.to_numeric(df_test[col].astype(str).str.replace(',', '.'), errors='coerce')

# 3. Fill NaNs with 0 (optional, but prevents errors in sums)
df_test[cols_to_process] = df_test[cols_to_process].fillna(0)


# 4. AGGREGATION PHASE: Group by 'porte' and sum
df_porte = df_test.groupby(['data_base','porte'])[cols_to_process].sum().reset_index()

# 5. CALCULATION PHASE: Now division will work!
df_porte['default_percent'] = (
    (df_porte['ativo_problematico'] / df_porte['carteira_ativa']) * 100
)

# 6. Convert Date
df_porte['data_base'] = pd.to_datetime(df_porte['data_base'])
df_porte['mes_ano'] = df_porte['data_base'].dt.strftime('%m/%Y')
df_porte = df_porte.drop('data_base', axis=1)

# 7. Alter Column order
ordered_cols = ['mes_ano', 'porte', 'vencido_acima_de_15_dias', 'carteira_inadimplida_arrastada', 'ativo_problematico', 'ativo_problematico', 'carteira_ativa', 'default_percent']
df_porte = df_porte[ordered_cols]

# Display results
display(df_porte)

,mes_ano,porte,vencido_acima_de_15_dias,carteira_inadimplida_arrastada,ativo_problematico,ativo_problematico,carteira_ativa,default_percent
0,04/2012,PF - Acima de 20 salários mínimos,2.335388e+09,3.749889e+09,6.740942e+09,6.740942e+09,1.291061e+11,5.221241
1,04/2012,PF - Até 1 salário mínimo,4.396522e+09,6.480802e+09,8.794702e+09,8.794702e+09,6.836709e+10,12.863941
2,04/2012,PF - Indisponível,1.635211e+08,4.164287e+08,5.481851e+08,5.481851e+08,7.648623e+09,7.167109
3,04/2012,PF - Mais de 1 a 2 salários mínimos,8.134761e+09,8.431810e+09,1.697862e+10,1.697862e+10,1.836521e+11,9.244988
4,04/2012,PF - Mais de 10 a 20 salários mínimos,1.486680e+09,2.114393e+09,3.948350e+09,3.948350e+09,7.333181e+10,5.384226
5,04/2012,PF - Mais de 2 a 3 salários mínimos,2.668145e+09,3.500402e+09,6.500042e+09,6.500042e+09,7.520056e+10,8.643609
6,04/2012,PF - Mais de 3 a 5 salários mínimos,2.770220e+09,3.758858e+09,6.619920e+09,6.619920e+09,9.097122e+10,7.276939
7,04/2012,PF - Mais de 5 a 10 salários mínimos,2.822133e+09,4.015228e+09,6.797094e+09,6.797094e+09,1.048307e+11,6.483879
8,04/2012,PF - Sem rendimento,9.761125e+09,1.392736e+10,1.915571e+10,1.915571e+10,1.961194e+11,9.767370
9,04/2012,PJ - Grande,1.730356e+09,2.442971e+09,6.357954e+09,6.357954e+09,5.094913e+11,1.247902


In [80]:
# 1. Setup paths
data_folder = '../data'
# Get all files starting with 'planilha_' and ending in '.csv'
all_files = sorted(glob.glob(os.path.join(data_folder, "planilha_*.csv")))
# 2. Filter files to start from 201204 to 202508
# 201201 to 201203 doesen't have column "ativo_problematico"
files_to_process = [
    f for f in all_files 
    if "planilha_" in f and "201204" <= f.split("planilha_")[-1].split(".")[0] <= "202508"
]
print(all_files)
print(files_to_process)

# List to store the 'df_porte' from each month
consolidated_data = []

print(f"Starting consolidation of {len(files_to_process)} files...")

for file_path in files_to_process:
    file_name = os.path.basename(file_path)
    
    try:
        # Load the month's data
        df_temp = pd.read_csv(file_path, sep=';', encoding='utf-8-sig')
        
        # --- LOGIC START ---
        
        # 1 & 2. Numeric Cleaning
        cols_to_process = ['vencido_acima_de_15_dias', 'carteira_inadimplida_arrastada', 'ativo_problematico', 'carteira_ativa']
        for col in cols_to_process:
            df_temp[col] = pd.to_numeric(df_temp[col].astype(str).str.replace(',', '.'), errors='coerce')
        
        # 3. Fill NaNs
        df_temp[cols_to_process] = df_temp[cols_to_process].fillna(0)
        
        # 4. Aggregation
        df_porte = df_temp.groupby(['data_base', 'porte'])[cols_to_process].sum().reset_index()
        
        # 5. Calculation
        df_porte['default_percent'] = (df_porte['ativo_problematico'] / df_porte['carteira_ativa']) * 100
        # Handle cases where carteira_ativa might be 0 to avoid Infinity
        df_porte['default_percent'] = df_porte['default_percent'].fillna(0)
        
        # 6. Date Conversion
        df_porte['data_base'] = pd.to_datetime(df_porte['data_base'])
        df_porte['mes_ano'] = df_porte['data_base'].dt.strftime('%m/%Y')
        
        # 7. Reorder and Select (dropped data_base implicitly by selecting ordered_cols)
        ordered_cols = ['mes_ano', 'porte', 'vencido_acima_de_15_dias', 'carteira_inadimplida_arrastada', 'ativo_problematico', 'carteira_ativa', 'default_percent']
        df_porte = df_porte[ordered_cols]
        
        
        # Append this month's summary to our list
        consolidated_data.append(df_porte)
        print(f"Done: {file_name}")
        
    except Exception as e:
        print(f"Error processing {file_name}: {e}")

# 3. Final Consolidation (The 'UNION ALL' equivalent)
df_final = pd.concat(consolidated_data, ignore_index=True)

# 4. Save the Master Worksheet
df_final.to_csv('../data/master_consolidation_2012_2025.csv', sep=';', index=False, encoding='utf-8-sig')

print("\n" + "="*30)
print(f"PROCESS COMPLETE!")
print(f"Total months processed: {len(consolidated_data)}")
print(f"Total rows in Master Worksheet: {len(df_final)}")
display(df_final.head())

['../data\\planilha_201201.csv', '../data\\planilha_201202.csv', '../data\\planilha_201203.csv', '../data\\planilha_201204.csv', '../data\\planilha_201205.csv', '../data\\planilha_201206.csv', '../data\\planilha_201207.csv', '../data\\planilha_201208.csv', '../data\\planilha_201209.csv', '../data\\planilha_201210.csv', '../data\\planilha_201211.csv', '../data\\planilha_201212.csv', '../data\\planilha_201301.csv', '../data\\planilha_201302.csv', '../data\\planilha_201303.csv', '../data\\planilha_201304.csv', '../data\\planilha_201305.csv', '../data\\planilha_201306.csv', '../data\\planilha_201307.csv', '../data\\planilha_201308.csv', '../data\\planilha_201309.csv', '../data\\planilha_201310.csv', '../data\\planilha_201311.csv', '../data\\planilha_201312.csv', '../data\\planilha_201401.csv', '../data\\planilha_201402.csv', '../data\\planilha_201403.csv', '../data\\planilha_201404.csv', '../data\\planilha_201405.csv', '../data\\planilha_201406.csv', '../data\\planilha_201407.csv', '../dat

KeyboardInterrupt: 

In [ ]:
# Part 1: 2012 to 2020
selic_part1 = sgs.get({'selic': 432}, start='2012-04-01', end='2019-12-31')

# Part 2: 2020 to 2025
selic_part2 = sgs.get({'selic': 432}, start='2020-01-01', end='2025-08-31')

# Combine them (Union)
selic = pd.concat([selic_part1, selic_part2])

# Reset index and create mes_ano
selic = selic.reset_index()
selic['mes_ano'] = selic['Date'].dt.strftime('%m/%Y')

# IMPORTANT: Since Selic 432 is daily, we take the LAST value of each month
# for your monthly TCC analysis
selic_monthly = selic.groupby('mes_ano').last().reset_index()

print("Full Range Loaded!")


Full Range Loaded!
     mes_ano       Date  selic
156  12/2020 2020-12-31   2.00
157  12/2021 2021-12-31   9.25
158  12/2022 2022-12-31  13.75
159  12/2023 2023-12-31  11.75
160  12/2024 2024-12-31  12.25


In [ ]:
selic_part3 = sgs.get({'selic': 432}, start='2025-01-01', end='2025-12-31')

In [ ]:
display(selic_monthly.sort_values("Date"))


,mes_ano,Date,selic
39,04/2012,2012-04-30,9.00
53,05/2012,2012-05-31,8.50
67,06/2012,2012-06-30,8.50
81,07/2012,2012-07-31,8.00
95,08/2012,2012-08-31,7.50
...,...,...,...
52,04/2025,2025-04-30,14.25
66,05/2025,2025-05-31,14.75
80,06/2025,2025-06-30,15.00
94,07/2025,2025-07-31,15.00


In [ ]:
# This is more reliable than setting system locales
pt_months_map = {
    'jan': '01', 'fev': '02', 'mar': '03', 'abr': '04', 
    'mai': '05', 'jun': '06', 'jul': '07', 'ago': '08', 
    'set': '09', 'out': '10', 'nov': '11', 'dez': '12'
}

# 2. Split the 'mes_ano' column (nov/12 -> ['nov', '12'])
df_final[['mes_temp', 'ano_temp']] = df_final['mes_ano'].str.split('/', expand=True)

# 3. Map the month name to its number
df_final['mes_num'] = df_final['mes_temp'].str.lower().map(pt_months_map)


In [ ]:
all_files = sorted(glob.glob(os.path.join(data_folder, "planilha_*.csv")))
all_files

['../data\\planilha_201201.csv',
 '../data\\planilha_201202.csv',
 '../data\\planilha_201203.csv',
 '../data\\planilha_201204.csv',
 '../data\\planilha_201205.csv',
 '../data\\planilha_201206.csv',
 '../data\\planilha_201207.csv',
 '../data\\planilha_201208.csv',
 '../data\\planilha_201209.csv',
 '../data\\planilha_201210.csv',
 '../data\\planilha_201211.csv',
 '../data\\planilha_201212.csv',
 '../data\\planilha_201301.csv',
 '../data\\planilha_201302.csv',
 '../data\\planilha_201303.csv',
 '../data\\planilha_201304.csv',
 '../data\\planilha_201305.csv',
 '../data\\planilha_201306.csv',
 '../data\\planilha_201307.csv',
 '../data\\planilha_201308.csv',
 '../data\\planilha_201309.csv',
 '../data\\planilha_201310.csv',
 '../data\\planilha_201311.csv',
 '../data\\planilha_201312.csv',
 '../data\\planilha_201401.csv',
 '../data\\planilha_201402.csv',
 '../data\\planilha_201403.csv',
 '../data\\planilha_201404.csv',
 '../data\\planilha_201405.csv',
 '../data\\planilha_201406.csv',
 '../data\

In [ ]:
data_folder2 = '../data'
file_name2 = 'master_consolidation_2012_2025.csv'
file_path2 = os.path.join(data_folder2, file_name2)

df_teste2 = pd.read_csv(file_path2, sep=';', encoding='utf-8-sig')

df_teste2

df_teste2.to_csv('../data/master_consolidation_2012_2025_2.csv', sep=';', index=False, encoding='utf-8-sig')

In [ ]:
data_folder2 = '../data'
file_name2 = 'master_consolidation_2012_2025.csv'
file_path2 = os.path.join(data_folder2, file_name2)

df_teste2 = pd.read_csv(file_path2, sep=';', encoding='utf-8-sig')

# 3. Define the Portuguese to Numeric mapping
pt_months_map = {
    'jan': '01', 'fev': '02', 'mar': '03', 'abr': '04', 
    'mai': '05', 'jun': '06', 'jul': '07', 'ago': '08', 
    'set': '09', 'out': '10', 'nov': '11', 'dez': '12'
}

# 4. Transform the 'mes_ano' column
# Splitting 'abr/12' -> ['abr', '12']
df_teste2[['mes_temp', 'ano_temp']] = df_teste2['mes_ano'].str.split('/', expand=True)

# Map the month name and rebuild as MM/YYYY
df_teste2['mes_ano'] = df_teste2['mes_temp'].str.lower().map(pt_months_map) + '/20' + df_teste2['ano_temp']


df_teste2.to_csv('../data/master_consolidation_2012_2025_2.csv', sep=';', index=False, encoding='utf-8-sig')


In [ ]:
data_folder3 = '../data'
file_name3 = 'master_consolidation_2012_2025_2.csv'
file_path3 = os.path.join(data_folder3, file_name3)
df_teste3 = pd.read_csv(file_path3, sep=';', encoding='utf-8-sig')

df_teste3

,mes_ano,porte,vencido_acima_de_15_dias,carteira_inadimplida_arrastada,ativo_problematico,carteira_ativa,default_percent,mes_temp,ano_temp
0,04/2012,PF - Acima de 20 salários mínimos,2335387900,3749888871,6740942473,"1,29E+11","5,221240611",abr,12
1,04/2012,PF - Até 1 salário mínimo,4396521762,6480802329,8794701984,68367090837,"12,86394064",abr,12
2,04/2012,PF - Indisponível,"163521092,6","416428709,2","548185119,5",7648623067,"7,167108572",abr,12
3,04/2012,PF - Mais de 1 a 2 salários mínimos,8134761144,8431810387,16978618920,"1,84E+11","9,244988172",abr,12
4,04/2012,PF - Mais de 10 a 20 salários mínimos,1486680340,2114393221,3948350399,73331809074,"5,384226094",abr,12
...,...,...,...,...,...,...,...,...,...
2170,08/2025,PJ - Grande,4344395395,4370505731,46919051235,"1,18E+12","3,988831049",ago,25
2171,08/2025,PJ - Indisponível,2394009715,2645703048,6855516782,99764920713,"6,871670657",ago,25
2172,08/2025,PJ - Micro,4515948686,6565929253,11682850838,89152256656,"13,10438039",ago,25
2173,08/2025,PJ - Médio,15227500402,25543313658,55498236770,"6,66E+11","8,328337636",ago,25


In [ ]:
# 2. PREPARING THE MACROECONOMIC DATAFRAME (SELIC)
# Standardizing the Selic date column to match the banking 'MM/YYYY' format
# This ensures the 'Key' is identical in both datasets
selic_monthly['mes_ano'] = selic_monthly['Date'].dt.strftime('%m/%Y')

In [ ]:
# 3. EXECUTING THE MERGE (LEFT JOIN)
# We use a 'left join' to preserve all records from the banking master file.
# The 'on' parameter specifies 'mes_ano' as our Foreign Key.
df_tcc_final = pd.merge(
    df_teste3, 
    selic_monthly[['mes_ano', 'selic']], 
    on='mes_ano', 
    how='left'
)

df_tcc_final

,mes_ano,porte,vencido_acima_de_15_dias,carteira_inadimplida_arrastada,ativo_problematico,carteira_ativa,default_percent,mes_temp,ano_temp,selic
0,04/2012,PF - Acima de 20 salários mínimos,2335387900,3749888871,6740942473,"1,29E+11","5,221240611",abr,12,9.0
1,04/2012,PF - Até 1 salário mínimo,4396521762,6480802329,8794701984,68367090837,"12,86394064",abr,12,9.0
2,04/2012,PF - Indisponível,"163521092,6","416428709,2","548185119,5",7648623067,"7,167108572",abr,12,9.0
3,04/2012,PF - Mais de 1 a 2 salários mínimos,8134761144,8431810387,16978618920,"1,84E+11","9,244988172",abr,12,9.0
4,04/2012,PF - Mais de 10 a 20 salários mínimos,1486680340,2114393221,3948350399,73331809074,"5,384226094",abr,12,9.0
...,...,...,...,...,...,...,...,...,...,...
2170,08/2025,PJ - Grande,4344395395,4370505731,46919051235,"1,18E+12","3,988831049",ago,25,15.0
2171,08/2025,PJ - Indisponível,2394009715,2645703048,6855516782,99764920713,"6,871670657",ago,25,15.0
2172,08/2025,PJ - Micro,4515948686,6565929253,11682850838,89152256656,"13,10438039",ago,25,15.0
2173,08/2025,PJ - Médio,15227500402,25543313658,55498236770,"6,66E+11","8,328337636",ago,25,15.0


In [ ]:
cols_to_drop = [c for c in ['mes_temp', 'ano_temp'] if c in df_tcc_final.columns]
cols_to_drop
df_banking = df_tcc_final.drop(columns=cols_to_drop)

In [ ]:
missing_count = df_banking['selic'].isnull().sum()
missing_count

0

In [ ]:
df_tcc_final.to_csv(
    '../data/master_tcc_final_consolidated.csv', 
    sep=';', 
    decimal=',', 
    index=False, 
    encoding='utf-8-sig'
)

In [137]:

import pandas as pd
import requests
import time
from io import StringIO

def get_sgs_robust(code, start_date, max_retries=3):
    url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{code}/dados?formato=json&dataInicial={start_date}"
    
    for attempt in range(max_retries):
        try:
            # We use a 60-second timeout to give the server plenty of time
            response = requests.get(url, timeout=60)
            
            # If the API returns a 200 (Success) but empty content, skip it
            if response.status_code == 200 and len(response.text.strip()) > 0:
                df = pd.read_json(StringIO(response.text))
                df['data'] = pd.to_datetime(df['data'], dayfirst=True)
                return df.set_index('data')
            
            print(f"Attempt {attempt+1}: Received empty or invalid response for code {code}. Retrying...")
            
        except Exception as e:
            print(f"Attempt {attempt+1} failed for code {code}: {e}")
        
        # Wait 2 seconds before the next attempt
        time.sleep(2)
    
    print(f"All {max_retries} attempts failed for code {code}.")
    return None

# Fetching your Extractive Industry indicator (21860)
df_extrativa = get_sgs_robust(21860, "01/04/2012")

if df_extrativa is not None:
    print(df_extrativa.tail())

Attempt 1 failed for code 21860: Expected object or value


KeyboardInterrupt: 

In [ ]:
# 1. Definir os tickers (Mapeamento direto para facilitar o rename)
tickers_map = {
    '^BVSP': 'ibovespa',
    '^DJI': 'dow_jones',
    '^IXIC': 'nasdaq'
}

# 2. Buscar os dados
# Removi o ['Adj Close'] do final da linha para tratar o DataFrame primeiro
df_raw = yf.download(list(tickers_map.keys()), start='2012-04-01', end='2026-05-10')

# 3. Selecionar 'Adj Close' com segurança (evitando o KeyError)
# Em versões novas, o yfinance pode retornar ['Adj Close', 'Ticker'] ou ['Ticker', 'Adj Close']
if 'Adj Close' in df_raw.columns.levels[0]:
    df_prices = df_raw['Adj Close']
else:
    # Caso a coluna se chame apenas 'Close'
    df_prices = df_raw['Close']

# 4. Renomear as colunas usando o mapeamento
df_prices = df_prices.rename(columns=tickers_map)

# 5. Transformar para MENSAL (O que seu HLM2 na USP exige)
df_mensal = df_prices.resample('M').last()

# 6. Criar a coluna mes_ano e limpar o índice
df_mensal = df_mensal.reset_index()
df_mensal['mes_ano'] = df_mensal['Date'].dt.strftime('%m/%Y')

# Opcional: remover a coluna 'Date' se você só for usar o 'mes_ano' para o merge
# df_mensal = df_mensal.drop(columns=['Date'])

display(df_mensal.tail())

In [16]:
# 1. Define your indicators
indicators = {
  #'selic': 432
  #'ipca': 433
  #'unemployment': 24369
  #'ibc_br': 24363
  #'usd': 1
  #'igp_m': 189
  #'pib_12m_real': 4382
  #'pib_12m_dolar': 4382 / 1 (USD) --> 4385
  #'prod_geral': 21859,
  #'prod_transformacao': 21861,
  #'prod_bens_capital': 21862,
  #'prod_bens_intermediarios': 21863,
  #'prod_bens_consumo_total': 21864,
  #'prod_bens_consumo_duraveis': 21865,
  #'prod_bens_consumo_semi_nao_duraveis': 21866,
  # 'ibc_br': 17439
  #'inpc': 188
  #'igp_di': 190
  #'incc' : 192
  #'ipc_fipe' : 193
  #'pre_produtor_amplo' : 225
  #'prod_automovel' : 1373
  #'vend_tot_aut' : 1378
  #'vend_interno_aut' : 1379
  #'vend_externo_aut' : 1380
  #'consum_deriv_gasolina' : 1393
  #'consum_deriv_glp' : 1394
  #'consum_deriv_tot' : 1398
  #'consum_energ_ele_resid' : 1403
  #'consum_energ_ele_ind': 1404
  #'consum_energ_ele_out': 1405
  #'consum_energ_ele_tot': 1406
  #'emp_for_tot': 28763
  #'emp_for_ind': 28764
  #'emp_for_cons_civ': 28766
  #'emp_for_comercio': 28767
  #'emp_for_serv': 28768
  #'emp_for_agro': 28765
  #'tot_trabalho_formal_pj_informal' : 24364
  #'salario_min' : 1619
  #'indic_confianca_consumi' : '4393'
  #'indic_cond_eco' : '4394'
  #'indic_expec_fut' : '4395'
  #'div_liq_set_publico_govfe_bacen': 2053,
  #'div_liq_set_publico_govfe': 2054,
  #'div_liq_set_publico_bacen': 2055,
  #'div_liq_set_publico_govest_munici': 2056,
  #'div_liq_set_publico_emp_est': 2059,
  #'div_liq_set_publico_govfe_bacen_interna': 2063,
  #'div_liq_set_publico_govest_munici_interna': 2066,
  #'div_liq_set_publico_empest_interna': 2069,
  #'div_liq_set_publico_govfe_bacen_ext': 2073,
  #'div_liq_set_publico_govest_munici_ext': 2076,
  #'div_liq_set_publico_empest_ext': 2079
  #'div_liq_set_publico_consolidado': 4003
  #'div_liq_set_publico_consolidado_interna': 4004
  #'div_liq_set_publico_consolidado_externa': 4005
  #'div_fis_liq' : 10819
  #'div_fis_liq_com_cambio' : 10829
  #'papel_moeda_emitido' : 1783
  #'reserva_bancaria' : 1784 
  #'reserva_bancaria' : 1784 
  #*************************************
  #Valores do yfinance
  #'^BVSP': 'ibovespa',
  #'^DJI': 'dow_jones',
  #'^IXIC': 'nasdaq'
 #*************************************
  #'comprometimento_renda' : 29037
  #'endividamento_familias' : 29033
  #'saldo_carteira_credito_total': 20539,
  #'saldo_carteira_credito_pj_total': 20540,
  #'saldo_carteira_credito_pf_total': 20541,
  #'credito_recursos_livres_pj_desconto_duplicatas': 20544,
  #'credito_recursos_livres_pj_desconto_cheques': 20545,
  #'credito_recursos_livres_pj_antecipacao_faturas_cartao': 20546,
  #'credito_recursos_livres_pj_capital_giro_total': 20550,
  #'credito_recursos_livres_pj_conta_garantida': 20551,
  #'credito_recursos_livres_pj_cheque_especial': 20552,
  #'credito_recursos_livres_pj_aquisicao_bens_total': 20555,
  #'credito_recursos_livres_pj_arrendamento_mercantil_total': 20558,
  #'credito_recursos_livres_pj_vendor': 20559,
  #'credito_recursos_livres_pj_compror': 20560 
  #'credito_recursos_livres_pj_cartao_credito_rotativo': 20561,
  #'credito_recursos_livres_pj_cartao_credito_parcelado': 20562,
  #'credito_recursos_livres_pj_cartao_credito_a_vista': 20563,
  #'credito_recursos_livres_pj_cartao_credito_total': 20564,
  #'credito_recursos_livres_pj_adiantamento_contratos_cambio_acc': 20565,
  #'credito_recursos_livres_pf_cheque_especial': 20573,
  #'credito_recursos_livres_pf_credito_pessoal_total': 20580,
  #'credito_recursos_livres_pf_aquisicao_bens_total': 20583,
  #'credito_recursos_livres_pf_arrendamento_mercantil_total': 20586,
  #'credito_recursos_livres_pf_cartao_credito_total': 20590,
  #'credito_recursos_livres_pf_desconto_cheques': 20591,
  #'credito_recursos_livres_pf_outros_creditos_livres': 20592,
  #'exp_total_usd': 22710,      
  #'exp_primarios_usd': 22711,  
  #'exp_semimanuf_usd': 22712,  
  #'exp_china_usd': 22716
  #'imp_total_usd': 22723,      
  #'imp_china_usd': 22729,      
  #'imp_usa_usd': 22727
  #"consumer_confidence": 4393
  #"base_mone_amp": 1831
  #"divid_mobiliria_saldos": 2213
  "pib_12m_dolar2": 432
}

# 2. Fetch data in two batches to bypass the 10-year limit
# Batch 1: 2012 to 2019
df_part1 = sgs.get(indicators, start='2012-04-01', end='2019-12-31')

# Batch 2: 2020 to present
df_part2 = sgs.get(indicators, start='2020-01-01')

# 3. Combine (Union) the batches
df_macro_full = pd.concat([df_part1, df_part2])


# IMPORTANT: Since Selic 432 is daily, we take the LAST value of each month
# for your monthly TCC analysis


# Reset index and create mes_ano
print(df_macro_full[df_macro_full.index == "2012-04-30"])
df_macro_full = df_macro_full.reset_index()
df_macro_full['mes_ano'] = df_macro_full['Date'].dt.strftime('%m/%Y')
cols_to_drop = [c for c in ['Date'] if c in df_macro_full.columns]
df_macro_full = df_macro_full.drop(columns=cols_to_drop)

#df_macro_full = df_macro_full.groupby('mes_ano').last().reset_index()
display(df_macro_full)

            pib_12m_dolar2
Date                      
2012-04-30             9.0


,pib_12m_dolar2,mes_ano
0,9.75,04/2012
1,9.75,04/2012
2,9.75,04/2012
3,9.75,04/2012
4,9.75,04/2012
...,...,...
5185,14.50,06/2026
5186,14.50,06/2026
5187,14.50,06/2026
5188,14.50,06/2026


In [6]:
data_folder2 = '../data'
file_name2 = 'master_tcc_final_consolidated.csv'
file_path2 = os.path.join(data_folder2, file_name2)

df_teste2 = pd.read_csv(file_path2, sep=';', encoding='utf-8-sig')
display(df_teste2)


df_tcc_final = pd.merge(
    df_teste2, 
    df_macro_full[['mes_ano', 
   'pib_12m_dolar2']], 
    on='mes_ano', 
    how='left'
)





display(df_tcc_final)

,mes_ano,porte,vencido_acima_de_15_dias,carteira_inadimplida_arrastada,ativo_problematico,carteira_ativa,default_percent,selic,ipca,unemployment,...,exp_total_usd,exp_primarios_usd,exp_semimanuf_usd,exp_china_usd,imp_total_usd,imp_china_usd,imp_usa_usd,consumer_confidence,base_mone_amp,divid_mobiliria_saldos
0,04/2012,PF - Acima de 20 salários mínimos,2335387900,3749888871,6740942473,"1,29E+11","5,221240611",9,"0,64","7,8",...,"513,6","19365,8","18852,3","149,1","0,0","311,4","1,8","164,99",1768640000,"2637606,46"
1,04/2012,PF - Até 1 salário mínimo,4396521762,6480802329,8794701984,68367090837,"12,86394064",9,"0,64","7,8",...,"513,6","19365,8","18852,3","149,1","0,0","311,4","1,8","164,99",1768640000,"2637606,46"
2,04/2012,PF - Indisponível,"163521092,6","416428709,2","548185119,5",7648623067,"7,167108572",9,"0,64","7,8",...,"513,6","19365,8","18852,3","149,1","0,0","311,4","1,8","164,99",1768640000,"2637606,46"
3,04/2012,PF - Mais de 1 a 2 salários mínimos,8134761144,8431810387,16978618920,"1,84E+11","9,244988172",9,"0,64","7,8",...,"513,6","19365,8","18852,3","149,1","0,0","311,4","1,8","164,99",1768640000,"2637606,46"
4,04/2012,PF - Mais de 10 a 20 salários mínimos,1486680340,2114393221,3948350399,73331809074,"5,384226094",9,"0,64","7,8",...,"513,6","19365,8","18852,3","149,1","0,0","311,4","1,8","164,99",1768640000,"2637606,46"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2170,08/2025,PJ - Grande,4344395395,4370505731,46919051235,"1,18E+12","3,988831049",15,"-0,11","5,6",...,"4753,4","29130,0","24376,6","538,7","1,4","615,5","23,2","111,88",7824002000,"10616189,85"
2171,08/2025,PJ - Indisponível,2394009715,2645703048,6855516782,99764920713,"6,871670657",15,"-0,11","5,6",...,"4753,4","29130,0","24376,6","538,7","1,4","615,5","23,2","111,88",7824002000,"10616189,85"
2172,08/2025,PJ - Micro,4515948686,6565929253,11682850838,89152256656,"13,10438039",15,"-0,11","5,6",...,"4753,4","29130,0","24376,6","538,7","1,4","615,5","23,2","111,88",7824002000,"10616189,85"
2173,08/2025,PJ - Médio,15227500402,25543313658,55498236770,"6,66E+11","8,328337636",15,"-0,11","5,6",...,"4753,4","29130,0","24376,6","538,7","1,4","615,5","23,2","111,88",7824002000,"10616189,85"


,mes_ano,porte,vencido_acima_de_15_dias,carteira_inadimplida_arrastada,ativo_problematico,carteira_ativa,default_percent,selic,ipca,unemployment,...,exp_primarios_usd,exp_semimanuf_usd,exp_china_usd,imp_total_usd,imp_china_usd,imp_usa_usd,consumer_confidence,base_mone_amp,divid_mobiliria_saldos,pib_12m_dolar2
0,04/2012,PF - Acima de 20 salários mínimos,2335387900,3749888871,6740942473,"1,29E+11","5,221240611",9,"0,64","7,8",...,"19365,8","18852,3","149,1","0,0","311,4","1,8","164,99",1768640000,"2637606,46",207866
1,04/2012,PF - Até 1 salário mínimo,4396521762,6480802329,8794701984,68367090837,"12,86394064",9,"0,64","7,8",...,"19365,8","18852,3","149,1","0,0","311,4","1,8","164,99",1768640000,"2637606,46",207866
2,04/2012,PF - Indisponível,"163521092,6","416428709,2","548185119,5",7648623067,"7,167108572",9,"0,64","7,8",...,"19365,8","18852,3","149,1","0,0","311,4","1,8","164,99",1768640000,"2637606,46",207866
3,04/2012,PF - Mais de 1 a 2 salários mínimos,8134761144,8431810387,16978618920,"1,84E+11","9,244988172",9,"0,64","7,8",...,"19365,8","18852,3","149,1","0,0","311,4","1,8","164,99",1768640000,"2637606,46",207866
4,04/2012,PF - Mais de 10 a 20 salários mínimos,1486680340,2114393221,3948350399,73331809074,"5,384226094",9,"0,64","7,8",...,"19365,8","18852,3","149,1","0,0","311,4","1,8","164,99",1768640000,"2637606,46",207866
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2170,08/2025,PJ - Grande,4344395395,4370505731,46919051235,"1,18E+12","3,988831049",15,"-0,11","5,6",...,"29130,0","24376,6","538,7","1,4","615,5","23,2","111,88",7824002000,"10616189,85",195382
2171,08/2025,PJ - Indisponível,2394009715,2645703048,6855516782,99764920713,"6,871670657",15,"-0,11","5,6",...,"29130,0","24376,6","538,7","1,4","615,5","23,2","111,88",7824002000,"10616189,85",195382
2172,08/2025,PJ - Micro,4515948686,6565929253,11682850838,89152256656,"13,10438039",15,"-0,11","5,6",...,"29130,0","24376,6","538,7","1,4","615,5","23,2","111,88",7824002000,"10616189,85",195382
2173,08/2025,PJ - Médio,15227500402,25543313658,55498236770,"6,66E+11","8,328337636",15,"-0,11","5,6",...,"29130,0","24376,6","538,7","1,4","615,5","23,2","111,88",7824002000,"10616189,85",195382


In [14]:
df_tcc_final.to_csv(
    '../data/master_tcc_final_consolidated.csv', 
    sep=';', 
    decimal=',', 
    index=False, 
    encoding='utf-8-sig'
)

In [15]:
data_folder2 = '../data'
file_name2 = 'master_tcc_final_consolidated.csv'
file_path2 = os.path.join(data_folder2, file_name2)

df_teste2 = pd.read_csv(file_path2, sep=';', encoding='utf-8-sig')
display(df_teste2)

,mes_ano,porte,vencido_acima_de_15_dias,carteira_inadimplida_arrastada,ativo_problematico,carteira_ativa,default_percent,selic,ipca,unemployment,...,exp_total_usd,exp_primarios_usd,exp_semimanuf_usd,exp_china_usd,imp_total_usd,imp_china_usd,imp_usa_usd,consumer_confidence,base_mone_amp,divid_mobiliria_saldos
0,04/2012,PF - Acima de 20 salários mínimos,2335387900,3749888871,6740942473,"1,29E+11","5,221240611",9,"0,64","7,8",...,"513,6","19365,8","18852,3","149,1","0,0","311,4","1,8","164,99",1768640000,"2637606,46"
1,04/2012,PF - Até 1 salário mínimo,4396521762,6480802329,8794701984,68367090837,"12,86394064",9,"0,64","7,8",...,"513,6","19365,8","18852,3","149,1","0,0","311,4","1,8","164,99",1768640000,"2637606,46"
2,04/2012,PF - Indisponível,"163521092,6","416428709,2","548185119,5",7648623067,"7,167108572",9,"0,64","7,8",...,"513,6","19365,8","18852,3","149,1","0,0","311,4","1,8","164,99",1768640000,"2637606,46"
3,04/2012,PF - Mais de 1 a 2 salários mínimos,8134761144,8431810387,16978618920,"1,84E+11","9,244988172",9,"0,64","7,8",...,"513,6","19365,8","18852,3","149,1","0,0","311,4","1,8","164,99",1768640000,"2637606,46"
4,04/2012,PF - Mais de 10 a 20 salários mínimos,1486680340,2114393221,3948350399,73331809074,"5,384226094",9,"0,64","7,8",...,"513,6","19365,8","18852,3","149,1","0,0","311,4","1,8","164,99",1768640000,"2637606,46"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2170,08/2025,PJ - Grande,4344395395,4370505731,46919051235,"1,18E+12","3,988831049",15,"-0,11","5,6",...,"4753,4","29130,0","24376,6","538,7","1,4","615,5","23,2","111,88",7824002000,"10616189,85"
2171,08/2025,PJ - Indisponível,2394009715,2645703048,6855516782,99764920713,"6,871670657",15,"-0,11","5,6",...,"4753,4","29130,0","24376,6","538,7","1,4","615,5","23,2","111,88",7824002000,"10616189,85"
2172,08/2025,PJ - Micro,4515948686,6565929253,11682850838,89152256656,"13,10438039",15,"-0,11","5,6",...,"4753,4","29130,0","24376,6","538,7","1,4","615,5","23,2","111,88",7824002000,"10616189,85"
2173,08/2025,PJ - Médio,15227500402,25543313658,55498236770,"6,66E+11","8,328337636",15,"-0,11","5,6",...,"4753,4","29130,0","24376,6","538,7","1,4","615,5","23,2","111,88",7824002000,"10616189,85"


In [317]:
df_teste2["consum_energ_ele_ind"]

0       15530
1       15530
2       15530
3       15530
4       15530
        ...  
2170    17044
2171    17044
2172    17044
2173    17044
2174    17044
Name: consum_energ_ele_ind, Length: 2175, dtype: int64

In [314]:
df_teste2.rename(columns={"energy_consumed" : "consum_energ_ele_ind",	"ven_interno_aut" : "vend_interno_aut","ven_externo_aut" : "vend_externo_aut"}, inplace=True)
display(df_teste2)

,mes_ano,porte,vencido_acima_de_15_dias,carteira_inadimplida_arrastada,ativo_problematico,carteira_ativa,default_percent,selic,ipca,unemployment,...,ipc_fipe,pre_produtor_amplo,prod_automovel,vend_tot_aut,vend_interno_aut,vend_externo_aut,consum_deriv_gasolina,consum_deriv_glp,consum_deriv_tot,consum_energ_ele_resid
0,04/2012,PF - Acima de 20 salários mínimos,2335387900,3749888871,6740942473,"1,29E+11","5,221240611",9,"0,64","7,8",...,"0,47","1,25",266275,249016,199506,49510,531,211,1837,9928
1,04/2012,PF - Até 1 salário mínimo,4396521762,6480802329,8794701984,68367090837,"12,86394064",9,"0,64","7,8",...,"0,47","1,25",266275,249016,199506,49510,531,211,1837,9928
2,04/2012,PF - Indisponível,"163521092,6","416428709,2","548185119,5",7648623067,"7,167108572",9,"0,64","7,8",...,"0,47","1,25",266275,249016,199506,49510,531,211,1837,9928
3,04/2012,PF - Mais de 1 a 2 salários mínimos,8134761144,8431810387,16978618920,"1,84E+11","9,244988172",9,"0,64","7,8",...,"0,47","1,25",266275,249016,199506,49510,531,211,1837,9928
4,04/2012,PF - Mais de 10 a 20 salários mínimos,1486680340,2114393221,3948350399,73331809074,"5,384226094",9,"0,64","7,8",...,"0,47","1,25",266275,249016,199506,49510,531,211,1837,9928
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2170,08/2025,PJ - Grande,4344395395,4370505731,46919051235,"1,18E+12","3,988831049",15,"-0,11","5,6",...,"0,04","0,35",247918,243978,185760,58218,573,244,2186,13959
2171,08/2025,PJ - Indisponível,2394009715,2645703048,6855516782,99764920713,"6,871670657",15,"-0,11","5,6",...,"0,04","0,35",247918,243978,185760,58218,573,244,2186,13959
2172,08/2025,PJ - Micro,4515948686,6565929253,11682850838,89152256656,"13,10438039",15,"-0,11","5,6",...,"0,04","0,35",247918,243978,185760,58218,573,244,2186,13959
2173,08/2025,PJ - Médio,15227500402,25543313658,55498236770,"6,66E+11","8,328337636",15,"-0,11","5,6",...,"0,04","0,35",247918,243978,185760,58218,573,244,2186,13959


In [ ]:
df_tcc_final['date_temp'] = pd.to_datetime(df_tcc_final['mes_ano'], format='%m/%Y')


pan_start = pd.to_datetime('02/2020', format='%m/%Y')
pan_end   = pd.to_datetime('05/2023', format='%m/%Y')
lockdown_mo = pd.to_datetime('05/2020', format='%m/%Y')

df_tcc_final['is_pandemic'] = ((df_tcc_final['date_temp'] >= pan_start) & (df_tcc_final['date_temp'] <= pan_end)).astype(int)

df_tcc_final['is_lockdown'] = (df_tcc_final['date_temp'] == lockdown_mo).astype(int)


display(df_tcc_final.loc[df_tcc_final['mes_ano'].isin(['04/2020', '05/2020', '06/2020', '01/2024'])])

In [13]:
data_folder2 = '../data'
file_name2 = 'master_tcc_final_consolidated.csv'
file_path2 = os.path.join(data_folder2, file_name2)

df_teste2 = pd.read_csv(file_path2, sep=';', encoding='utf-8-sig')
display(df_teste2)


df_tcc_final = df_teste2


df_tcc_final = df_tcc_final.drop(columns=['pib_12m_dolar2'])

print('pib_12m_dolar2' in df_tcc_final.columns)

display(df_tcc_final)

,mes_ano,porte,vencido_acima_de_15_dias,carteira_inadimplida_arrastada,ativo_problematico,carteira_ativa,default_percent,selic,ipca,unemployment,...,exp_primarios_usd,exp_semimanuf_usd,exp_china_usd,imp_total_usd,imp_china_usd,imp_usa_usd,consumer_confidence,base_mone_amp,divid_mobiliria_saldos,pib_12m_dolar2
0,04/2012,PF - Acima de 20 salários mínimos,2335387900,3749888871,6740942473,"1,29E+11","5,221240611",9,"0,64","7,8",...,"19365,8","18852,3","149,1","0,0","311,4","1,8","164,99",1768640000,"2637606,46",207866
1,04/2012,PF - Até 1 salário mínimo,4396521762,6480802329,8794701984,68367090837,"12,86394064",9,"0,64","7,8",...,"19365,8","18852,3","149,1","0,0","311,4","1,8","164,99",1768640000,"2637606,46",207866
2,04/2012,PF - Indisponível,"163521092,6","416428709,2","548185119,5",7648623067,"7,167108572",9,"0,64","7,8",...,"19365,8","18852,3","149,1","0,0","311,4","1,8","164,99",1768640000,"2637606,46",207866
3,04/2012,PF - Mais de 1 a 2 salários mínimos,8134761144,8431810387,16978618920,"1,84E+11","9,244988172",9,"0,64","7,8",...,"19365,8","18852,3","149,1","0,0","311,4","1,8","164,99",1768640000,"2637606,46",207866
4,04/2012,PF - Mais de 10 a 20 salários mínimos,1486680340,2114393221,3948350399,73331809074,"5,384226094",9,"0,64","7,8",...,"19365,8","18852,3","149,1","0,0","311,4","1,8","164,99",1768640000,"2637606,46",207866
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2170,08/2025,PJ - Grande,4344395395,4370505731,46919051235,"1,18E+12","3,988831049",15,"-0,11","5,6",...,"29130,0","24376,6","538,7","1,4","615,5","23,2","111,88",7824002000,"10616189,85",195382
2171,08/2025,PJ - Indisponível,2394009715,2645703048,6855516782,99764920713,"6,871670657",15,"-0,11","5,6",...,"29130,0","24376,6","538,7","1,4","615,5","23,2","111,88",7824002000,"10616189,85",195382
2172,08/2025,PJ - Micro,4515948686,6565929253,11682850838,89152256656,"13,10438039",15,"-0,11","5,6",...,"29130,0","24376,6","538,7","1,4","615,5","23,2","111,88",7824002000,"10616189,85",195382
2173,08/2025,PJ - Médio,15227500402,25543313658,55498236770,"6,66E+11","8,328337636",15,"-0,11","5,6",...,"29130,0","24376,6","538,7","1,4","615,5","23,2","111,88",7824002000,"10616189,85",195382


False


,mes_ano,porte,vencido_acima_de_15_dias,carteira_inadimplida_arrastada,ativo_problematico,carteira_ativa,default_percent,selic,ipca,unemployment,...,exp_total_usd,exp_primarios_usd,exp_semimanuf_usd,exp_china_usd,imp_total_usd,imp_china_usd,imp_usa_usd,consumer_confidence,base_mone_amp,divid_mobiliria_saldos
0,04/2012,PF - Acima de 20 salários mínimos,2335387900,3749888871,6740942473,"1,29E+11","5,221240611",9,"0,64","7,8",...,"513,6","19365,8","18852,3","149,1","0,0","311,4","1,8","164,99",1768640000,"2637606,46"
1,04/2012,PF - Até 1 salário mínimo,4396521762,6480802329,8794701984,68367090837,"12,86394064",9,"0,64","7,8",...,"513,6","19365,8","18852,3","149,1","0,0","311,4","1,8","164,99",1768640000,"2637606,46"
2,04/2012,PF - Indisponível,"163521092,6","416428709,2","548185119,5",7648623067,"7,167108572",9,"0,64","7,8",...,"513,6","19365,8","18852,3","149,1","0,0","311,4","1,8","164,99",1768640000,"2637606,46"
3,04/2012,PF - Mais de 1 a 2 salários mínimos,8134761144,8431810387,16978618920,"1,84E+11","9,244988172",9,"0,64","7,8",...,"513,6","19365,8","18852,3","149,1","0,0","311,4","1,8","164,99",1768640000,"2637606,46"
4,04/2012,PF - Mais de 10 a 20 salários mínimos,1486680340,2114393221,3948350399,73331809074,"5,384226094",9,"0,64","7,8",...,"513,6","19365,8","18852,3","149,1","0,0","311,4","1,8","164,99",1768640000,"2637606,46"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2170,08/2025,PJ - Grande,4344395395,4370505731,46919051235,"1,18E+12","3,988831049",15,"-0,11","5,6",...,"4753,4","29130,0","24376,6","538,7","1,4","615,5","23,2","111,88",7824002000,"10616189,85"
2171,08/2025,PJ - Indisponível,2394009715,2645703048,6855516782,99764920713,"6,871670657",15,"-0,11","5,6",...,"4753,4","29130,0","24376,6","538,7","1,4","615,5","23,2","111,88",7824002000,"10616189,85"
2172,08/2025,PJ - Micro,4515948686,6565929253,11682850838,89152256656,"13,10438039",15,"-0,11","5,6",...,"4753,4","29130,0","24376,6","538,7","1,4","615,5","23,2","111,88",7824002000,"10616189,85"
2173,08/2025,PJ - Médio,15227500402,25543313658,55498236770,"6,66E+11","8,328337636",15,"-0,11","5,6",...,"4753,4","29130,0","24376,6","538,7","1,4","615,5","23,2","111,88",7824002000,"10616189,85"
